In [1]:
import numpy as np
import cv2

class KalmanFilter6D:
    def __init__(self, dt, process_variance, measurement_variance):

        self.kf = cv2.KalmanFilter(14, 7)  # [x,y,z, qx, qy, qz, qw, vx, vy, vz, vqx, vqy, vqz, vqw], [x,y,z, qx, qy, qz, qw]
        self.dt = dt

        self.kf.transitionMatrix = np.eye(14, dtype=np.float32)
        for i in range(7):
            self.kf.transitionMatrix[i, i+7] = dt
        

        self.kf.measurementMatrix = np.zeros((7, 14), dtype=np.float32)

        for i in range(7):
            self.kf.measurementMatrix[i, i] = 1
        

        self.kf.processNoiseCov = np.eye(14, dtype=np.float32) * process_variance

        

        self.kf.measurementNoiseCov = np.eye(7, dtype=np.float32) * measurement_variance
        
        self.previous_measurement = None

    def predict(self):
        return self.kf.predict()

    def correct(self, measurement):
        if self.previous_measurement is not None:
            velocity_estimate = (measurement - self.previous_measurement) / self.dt
            velocity_estimate = velocity_estimate[:, np.newaxis]
            self.kf.statePost[7:14] = velocity_estimate
        self.previous_measurement = measurement.copy()
        return self.kf.correct(measurement)



In [2]:
banana_transform = np.loadtxt("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/world -> banana.txt",dtype= np.float32)

In [ ]:
def nothing_print():
    print("nothing")
    return

In [ ]:
# Example usage:

# Define time difference between measurements, process and measurement variances based on your needs
dt = 1.0 / 5.0  # since it's 5 measurements per second
kf_6d = KalmanFilter6D(dt, process_variance=1e-3, measurement_variance=1e-2)

# Let's assume you get a new measurement from your pose estimation algorithm as follows
# new_measurement = np.array([x, y, z, qx, qy, qz, qw], dtype=np.float32)
states = []

for index in np.arange(banana_transform.shape[0]):
    # Predict step
    prediction = kf_6d.predict()

    # Correct step
    updated_state = kf_6d.correct(banana_transform[index])
    states.append(updated_state.flatten())
    print(updated_state.flatten())
    print("\n")

states = np.asarray(states)
np.savetxt("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/banana_transform_calman.txt", states)


In [1]:
import numpy as np
import cv2

class KalmanFilter6DWithAcceleration:
    def __init__(self, dt, process_variance, measurement_variance):
        # Define Kalman filter
        self.kf = cv2.KalmanFilter(21, 7)
        
        # State transition matrix
        self.kf.transitionMatrix = np.eye(21, dtype=np.float32)
        for i in range(7):
            self.kf.transitionMatrix[i, i+7] = dt
            self.kf.transitionMatrix[i, i+14] = 0.5 * dt**2
            self.kf.transitionMatrix[i+7, i+14] = dt
        
        # Measurement matrix
        self.kf.measurementMatrix = np.zeros((7, 21), dtype=np.float32)
        for i in range(7):
            self.kf.measurementMatrix[i, i] = 1
        
        # Process noise covariance matrix
        self.kf.processNoiseCov = np.eye(21, dtype=np.float32) * process_variance
        
        # Measurement noise covariance matrix
        self.kf.measurementNoiseCov = np.eye(7, dtype=np.float32) * measurement_variance
        
        self.previous_measurement = None
        self.previous_velocity = None

    def predict(self):
        return self.kf.predict()

    def correct(self, measurement):
        if self.previous_measurement is not None:
            velocity_estimate = (measurement - self.previous_measurement) / dt
            if self.previous_velocity is not None:
                acceleration_estimate = (velocity_estimate - self.previous_velocity) / dt
                self.kf.statePost[14:21, 0] = acceleration_estimate[:, np.newaxis].reshape(7)
            self.kf.statePost[7:14, 0] = velocity_estimate[:, np.newaxis].reshape(7)
            self.previous_velocity = velocity_estimate
        self.previous_measurement = measurement.copy()
        return self.kf.correct(measurement).flatten()[:7]  # Only return the filtered pose values

# Example usage:

dt = 1.0 / 5.0  # since it's 5 measurements per second
kf_6d_with_accel = KalmanFilter6DWithAcceleration(dt, process_variance=1e-3, measurement_variance=1e-2)

# Assume you have n pose estimations in a matrix
banana_transform = np.loadtxt("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/world -> banana.txt",dtype= np.float32)
# pose_matrix = np.random.rand(100, 7)  # example, replace this with your data

filtered_poses = []
for idx in range(banana_transform.shape[0]):
    new_measurement = banana_transform[idx, :]
    prediction = kf_6d_with_accel.predict()
    filtered_pose = kf_6d_with_accel.correct(new_measurement)
    filtered_poses.append(filtered_pose)

filtered_poses = np.array(filtered_poses)
np.savetxt("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/filtered_poses.txt",filtered_poses)
